# CLAMPfull pseudobulk — LV biology validation

For each pseudobulk dataset, find the per-dataset CLAMPfull LVs that correlate most
with ground-truth cell-type fractions, then validate their biology:

1. Compute Pearson correlation between CLAMPfull B matrix and truth fractions
2. Select top-N LVs per cell type (r ≥ threshold)
3. Extract significant pathways from the CLAMP summary
4. Run CellMarker ORA on top-1% gene loadings
5. Gene loading scatter plots with top-gene labels

💡 **Environment:** `clamp-analyses`

## Libraries

In [1]:
library(here)
library(dplyr)
library(tidyr)
library(ggplot2)
library(ggrepel)
library(patchwork)
library(stringr)
library(Matrix)

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union





Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack




## Settings

In [2]:
DATASETS     <- c("Brain_Mathys2023", "Brain_Xiong2023", "BRCA_Bassez2021","CRC_Pelka2021", "PBMC_1k1k", "PBMC_Perez2022")
MOD_ROOT     <- here("output", "01_model_building", "05_pseudobulk")
DATA_ROOT    <- here("data", "pseudobulk")
CM_PATH      <- here("data", "archs4", "CellMarker_pathMat.rds")
OUT_DIR      <- here("output", "03_model_biology", "02_pseudobulk", "01_biology")
FDR_PATH     <- 0.25     # pathway FDR threshold (CLAMP summary)
AUC_PATH     <- 0.60     # pathway AUC threshold
FDR_CM       <- 0.25     # CellMarker ORA FDR threshold
TOP_GENE_PCT <- 0.01     # top 1% gene loadings for ORA and plots
N_LABEL      <- 10       # number of genes to label on loading plots

dir.create(file.path(OUT_DIR, "plots"), recursive = TRUE, showWarnings = FALSE)
set.seed(42)

# Cell type display labels (short code -> full name for plots)
ct_labels_df <- read.csv(here("data", "pseudobulk", "cell_type_labels.csv"),
                         stringsAsFactors = FALSE)
CT_LABELS    <- setNames(ct_labels_df$label, ct_labels_df$cell_type)
ct_label     <- function(x) ifelse(x %in% names(CT_LABELS), CT_LABELS[x], x)

## Load CellMarker gene sets

In [3]:
cm_mat      <- readRDS(CM_PATH)   # dgCMatrix: genes x cell-type-marker sets
cm_t2g      <- lapply(colnames(cm_mat), function(s) rownames(cm_mat)[cm_mat[, s] != 0])
names(cm_t2g) <- colnames(cm_mat)
all_cm_genes  <- rownames(cm_mat)

cat("CellMarker gene sets:", length(cm_t2g), "\n")
cat("Unique genes:",         length(all_cm_genes), "\n")

CellMarker gene sets: 1692 
Unique genes: 12642 


In [4]:
# Load gene set databases
# c8  — MSigDB cell-type signatures    (all datasets)
# c7  — MSigDB immunological signatures (immune datasets)
# Allen Brain Atlas 10x scRNA 2021     (brain datasets)
C7_PATH    <- here("data", "pathways", "c7.all.v2026.1.Hs.symbols.gmt")
C8_PATH    <- here("data", "pathways", "c8.all.v2026.1.Hs.symbols.gmt")
ALLEN_PATH <- here("data", "pathways", "Allen_Brain_Atlas_10x_scRNA_2021.gmt")

parse_gmt <- function(path) {
    lines  <- readLines(path)
    result <- vector("list", length(lines))
    nms    <- character(length(lines))
    for (i in seq_along(lines)) {
        parts    <- strsplit(lines[i], "\t")[[1]]
        nms[i]   <- parts[1]
        result[[i]] <- parts[-(1:2)]
    }
    names(result) <- nms
    result
}

c7_t2g    <- parse_gmt(C7_PATH)
c8_t2g    <- parse_gmt(C8_PATH)
allen_raw  <- parse_gmt(ALLEN_PATH)
allen_t2g  <- allen_raw[grepl(" up$", names(allen_raw))]   # marker genes only

cat("C7:", length(c7_t2g), "sets\n")
cat("C8:", length(c8_t2g), "sets\n")
cat("Allen Brain Atlas (up):", length(allen_t2g), "sets\n")

BRAIN_DS  <- c("Brain_Mathys2023", "Brain_Xiong2023")
IMMUNE_DS <- c("PBMC_1k1k", "PBMC_Perez2022", "BRCA_Bassez2021", "CRC_Pelka2021")

# Term label cleaners
clean_c8 <- function(x) {
    sapply(x, function(s) {
        parts <- strsplit(s, "_")[[1]]
        body  <- if (length(parts) > 3) paste(parts[-(1:2)], collapse = " ")
                 else gsub("_", " ", s)
        stringr::str_trunc(tools::toTitleCase(tolower(body)), 55)
    }, USE.NAMES = FALSE)
}

clean_c7 <- function(x) {
    x <- sub("^GSE[0-9]+_", "", x)
    x <- gsub("_VS_", " vs ", x)
    x <- sub("_(UP|DN)$", "", x)
    x <- gsub("_", " ", x)
    stringr::str_trunc(tools::toTitleCase(tolower(x)), 55)
}

clean_allen <- function(x) {
    x <- sub(" up$", "", x)
    x <- sub("^(Human|Mouse) ", "", x)
    stringr::str_trunc(x, 55)
}

Warning message in readLines(path):
“incomplete final line found on '/home/msubirana/Documents/pivlab/clamp-analyses/data/pathways/Allen_Brain_Atlas_10x_scRNA_2021.gmt'”


C7: 5219 sets
C8: 866 sets
Allen Brain Atlas (up): 383 sets


## CLAMPfull LV → cell-type assignments (from benchmark)

In [5]:
ASSIGNMENTS_PATH <- here("output", "03_model_biology", "02_pseudobulk",
                         "clampfull_lv_assignments.csv")

top_lvs_df <- read.csv(ASSIGNMENTS_PATH, stringsAsFactors = FALSE)

# Keep only configured datasets (respects DATASETS list; LUAD excluded)
top_lvs_df <- top_lvs_df[top_lvs_df$dataset %in% DATASETS, ]
rownames(top_lvs_df) <- NULL

cat("CLAMPfull LV assignments loaded:", nrow(top_lvs_df), "rows\n")
cat("Datasets:\n")
print(table(top_lvs_df$dataset))
cat("\nSample (first 10 rows):\n")
print(head(top_lvs_df, 10))

CLAMPfull LV assignments loaded: 35 rows
Datasets:

Brain_Mathys2023  Brain_Xiong2023    CRC_Pelka2021        PBMC_1k1k 
               7                7                8                9 
  PBMC_Perez2022 
               4 

Sample (first 10 rows):
   cell_type   LV       cor          dataset
1        Ast LV17 0.8911263 Brain_Mathys2023
2        Exc LV11 0.6028913 Brain_Mathys2023
3        Inh  LV4 0.8155799 Brain_Mathys2023
4        Mic  LV6 0.9655128 Brain_Mathys2023
5        Oli LV18 0.9617558 Brain_Mathys2023
6        Opc  LV7 0.4715819 Brain_Mathys2023
7        Vas LV12 0.3293936 Brain_Mathys2023
8        Ast  LV5 0.9427029  Brain_Xiong2023
9        Exc LV13 0.7277295  Brain_Xiong2023
10       Inh  LV2 0.6067130  Brain_Xiong2023


## Pathway extraction (CLAMP summary)

In [6]:
pathway_list <- lapply(DATASETS, function(ds) {
    sum_path  <- file.path(MOD_ROOT, ds, "CLAMPfull", "summary.csv")
    sum_df    <- read.csv(sum_path, stringsAsFactors = FALSE)
    lvs_in_ds <- unique(top_lvs_df$LV[top_lvs_df$dataset == ds])
    if (length(lvs_in_ds) == 0) return(NULL)

    sum_df %>%
        dplyr::filter(LV %in% lvs_in_ds, AUC >= AUC_PATH, FDR < FDR_PATH) %>%
        dplyr::mutate(
            dataset  = ds,
            pw_clean = gsub("^(BP_|C2CP_|CellMarker_)", "", pathway),
            pw_clean = gsub(" \\(GO:[0-9]+\\)", "", pw_clean),
            type     = dplyr::case_when(
                grepl("^BP_",         pathway) ~ "GO-BP",
                grepl("^CellMarker_", pathway) ~ "CellMarker",
                TRUE                           ~ "KEGG"
            )
        ) %>%
        dplyr::select(dataset, LV, pathway, pw_clean, type, AUC, FDR, npos, nneg)
})

lv_pathways_df <- do.call(rbind, Filter(Negate(is.null), pathway_list))
rownames(lv_pathways_df) <- NULL

cat("Pathway associations:", nrow(lv_pathways_df), "\n")
cat("Type breakdown:\n")
print(table(lv_pathways_df$type))

Pathway associations: 76 
Type breakdown:

GO-BP 
   76 


## ORA — C8 (all) · C7 (immune) · Allen Brain Atlas (brain)

In [7]:
run_ora <- function(query_genes, t2g, background_genes, min_gs = 5, max_gs = 500) {
    bg_set <- intersect(background_genes, unique(unlist(t2g)))
    q_set  <- intersect(query_genes, bg_set)
    N      <- length(bg_set)
    K      <- length(q_set)
    if (K == 0) return(NULL)

    res_list <- lapply(names(t2g), function(gs_name) {
        gs <- intersect(t2g[[gs_name]], bg_set)
        M  <- length(gs)
        if (M < min_gs || M > max_gs) return(NULL)
        k  <- length(intersect(q_set, gs))
        if (k == 0) return(NULL)
        data.frame(
            term            = gs_name,
            k               = k, K = K, M = M, N = N,
            fold_enrichment = (k / K) / (M / N),
            pvalue          = phyper(k - 1L, M, N - M, K, lower.tail = FALSE),
            stringsAsFactors = FALSE
        )
    })

    res_df <- do.call(rbind, Filter(Negate(is.null), res_list))
    if (is.null(res_df) || nrow(res_df) == 0) return(NULL)
    res_df$padj <- p.adjust(res_df$pvalue, method = "BH")
    dplyr::arrange(res_df, padj)
}

In [ ]:
ora_all <- lapply(DATASETS, function(ds) {
    ds_rows <- top_lvs_df[top_lvs_df$dataset == ds, ]
    if (nrow(ds_rows) == 0) return(NULL)

    Z_path      <- file.path(MOD_ROOT, ds, "CLAMPfull", "Z.csv")
    Z           <- read.csv(Z_path, row.names = 1, check.names = FALSE)
    model_genes <- rownames(Z)   # background = genes in the model

    dbs <- list(c8 = c8_t2g)
    if (ds %in% BRAIN_DS)  dbs[["allen"]] <- allen_t2g
    if (ds %in% IMMUNE_DS) dbs[["c7"]]    <- c7_t2g

    cat("ORA:", ds, "[", paste(names(dbs), collapse = "+"), "] —",
        nrow(Z), "genes,", nrow(ds_rows), "LVs\n")

    do.call(rbind, lapply(unique(ds_rows$LV), function(lv) {
        if (!lv %in% colnames(Z)) return(NULL)

        loadings  <- Z[[lv]]; names(loadings) <- rownames(Z)
        n_top     <- max(1L, ceiling(length(loadings) * TOP_GENE_PCT))
        top_genes <- names(sort(loadings, decreasing = TRUE))[seq_len(n_top)]
        ct        <- paste(ds_rows$cell_type[ds_rows$LV == lv], collapse = ", ")

        do.call(rbind, lapply(names(dbs), function(db_nm) {
            res <- run_ora(top_genes, dbs[[db_nm]], model_genes, min_gs = 10)
            if (is.null(res)) return(NULL)
            res %>%
                dplyr::filter(padj < FDR_CM) %>%
                dplyr::mutate(dataset = ds, cell_type = ct, LV = lv, db = db_nm) %>%
                dplyr::select(dataset, cell_type, LV, db, term, k, K, M, N,
                              fold_enrichment, pvalue, padj)
        }))
    }))
})

lv_ora_df <- do.call(rbind, Filter(Negate(is.null), ora_all))
if (!is.null(lv_ora_df)) rownames(lv_ora_df) <- NULL

cat("\nORA hits by database:\n")
print(table(lv_ora_df$db, lv_ora_df$dataset))

## Pathway bubble plots (GO-BP and KEGG)

In [ ]:
clean_pw <- function(x) {
    x <- gsub("^(BP_|C2CP_|CellMarker_)", "", x)
    x <- gsub(" \\(GO:[0-9]+\\)", "", x)
    x <- gsub("_", " ", x)
    stringr::str_trunc(x, 55)
}

bubble_theme <- theme(
    panel.background   = element_rect(fill = "white", color = NA),
    plot.background    = element_rect(fill = "white", color = NA),
    panel.grid.major.x = element_line(linetype = "dotted", linewidth = 0.8, color = "#cccccc"),
    panel.grid.major.y = element_blank(),
    panel.grid.minor   = element_blank(),
    axis.line.x        = element_line(color = "#333333", linewidth = 0.5),
    axis.line.y        = element_blank(),
    axis.ticks.y       = element_blank(),
    axis.text.y        = element_text(size = 10, hjust = 1),
    axis.text.x        = element_text(size = 10),
    axis.title.x       = element_text(size = 13, margin = margin(t = 12)),
    plot.title         = element_text(size = 13, hjust = 0.5, face = "bold",
                                      margin = margin(b = 10)),
    legend.position    = "top",
    legend.title       = element_text(size = 10),
    legend.text        = element_text(size = 9),
    plot.margin        = margin(10, 20, 10, 10)
)
green_colors <- c("#d9e6e2", "#4bc17c", "#007a33")

if (!is.null(lv_pathways_df) && nrow(lv_pathways_df) > 0) {

    pw_top <- lv_pathways_df %>%
        dplyr::mutate(
            pw_clean    = clean_pw(pathway),
            neg_log_fdr = pmin(-log10(FDR + 1e-300), 4),
            bubble_size = 2 + ((AUC - AUC_PATH) / (1 - AUC_PATH)) * 8
        ) %>%
        dplyr::group_by(dataset, LV) %>%
        dplyr::slice_max(AUC, n = 5, with_ties = FALSE) %>%
        dplyr::ungroup()

    for (ds in DATASETS) {
        ds_lvs <- unique(pw_top$LV[pw_top$dataset == ds])
        if (length(ds_lvs) == 0) { cat(ds, "— no pathways\n"); next }

        for (lv in ds_lvs) {
            d <- pw_top %>%
                dplyr::filter(dataset == ds, LV == lv) %>%
                dplyr::arrange(AUC) %>%
                dplyr::mutate(label = factor(pw_clean, levels = unique(pw_clean)))
            if (nrow(d) == 0) next

            ct_str  <- paste(ct_label(unique(top_lvs_df$cell_type[
                                top_lvs_df$dataset == ds & top_lvs_df$LV == lv])),
                             collapse = ", ")
            fdr_max <- max(2, ceiling(max(d$neg_log_fdr, na.rm = TRUE)))
            fig_h   <- max(4.0, 2.0 + 0.55 * nrow(d))
            options(repr.plot.width = 12, repr.plot.height = fig_h)

            p <- ggplot(d, aes(x = AUC, y = label)) +
                geom_segment(
                    aes(x = AUC_PATH, xend = AUC, yend = label),
                    color = "#cccccc", linewidth = 0.8, linetype = "dotted"
                ) +
                geom_point(aes(color = neg_log_fdr, size = bubble_size)) +
                scale_color_gradientn(
                    colors = green_colors, limits = c(0, fdr_max),
                    name   = expression("-" * log[10] ~ FDR)
                ) +
                scale_size_identity(guide = "none") +
                scale_x_continuous(
                    limits = c(AUC_PATH, 1.02),
                    breaks = seq(0.6, 1.0, by = 0.1)
                ) +
                labs(
                    x     = "AUC",
                    y     = NULL,
                    title = paste0(ds, " | ", lv, " | ", ct_str)
                ) +
                bubble_theme +
                guides(color = guide_colorbar(
                    barwidth = 7, barheight = 0.5,
                    title.position = "left", title.hjust = 1
                ))

            print(p)
        }
    }
} else {
    cat("No pathways passed thresholds (AUC ≥", AUC_PATH, ", FDR <", FDR_PATH, ")\n")
}

## ORA bubble plots (C8 · C7 · Allen Brain)

In [ ]:
clean_ora_term <- function(term, db) {
    dplyr::case_when(
        db == "allen" ~ clean_allen(term),
        db == "c7"    ~ clean_c7(term),
        TRUE          ~ clean_c8(term)
    )
}

if (!is.null(lv_ora_df) && nrow(lv_ora_df) > 0) {

    # preferred db per dataset type: allen for brain, c8 for others
    pref_db <- function(ds) if (ds %in% BRAIN_DS) "allen" else "c8"

    ora_plot_df <- lv_ora_df %>%
        dplyr::mutate(
            term_clean   = clean_ora_term(term, db),
            neg_log_padj = pmin(-log10(padj + 1e-300), 4),
            bubble_size  = 2 + pmin(fold_enrichment / 30, 1) * 8
        ) %>%
        dplyr::group_by(dataset, LV, db) %>%
        dplyr::slice_min(padj, n = 8, with_ties = FALSE) %>%
        dplyr::ungroup()

    for (ds in DATASETS) {
        preferred <- pref_db(ds)
        ds_lvs <- top_lvs_df$LV[top_lvs_df$dataset == ds]
        if (length(ds_lvs) == 0) next

        for (lv in unique(ds_lvs)) {
            ct_str  <- paste(ct_label(unique(top_lvs_df$cell_type[
                                top_lvs_df$dataset == ds & top_lvs_df$LV == lv])),
                             collapse = ", ")

            # build one panel per available db for this LV
            available_dbs <- unique(lv_ora_df$db[lv_ora_df$dataset == ds &
                                                  lv_ora_df$LV == lv])
            # show preferred db first
            show_dbs <- c(preferred,
                          setdiff(available_dbs, preferred))
            show_dbs <- show_dbs[show_dbs %in% available_dbs]
            if (length(show_dbs) == 0) next

            panels <- lapply(show_dbs, function(db_nm) {
                d <- ora_plot_df %>%
                    dplyr::filter(dataset == ds, LV == lv, db == db_nm) %>%
                    dplyr::arrange(neg_log_padj) %>%
                    dplyr::mutate(label = factor(term_clean, levels = unique(term_clean)))
                if (nrow(d) == 0) return(NULL)

                v_max <- max(2, ceiling(max(d$neg_log_padj, na.rm = TRUE)))
                ggplot(d, aes(x = neg_log_padj, y = label)) +
                    geom_segment(aes(x = 0, xend = neg_log_padj, yend = label),
                                 color = "#cccccc", linewidth = 0.8, linetype = "dotted") +
                    geom_point(aes(color = neg_log_padj, size = bubble_size)) +
                    scale_color_gradientn(colors = green_colors, limits = c(0, v_max),
                                          name = expression("-" * log[10] ~ "adj.p")) +
                    scale_size_identity(guide = "none") +
                    scale_x_continuous(limits = c(0, v_max + 0.3)) +
                    labs(x = expression("-" * log[10] ~ "adj.p"), y = NULL,
                         title = toupper(db_nm)) +
                    bubble_theme +
                    guides(color = guide_colorbar(barwidth = 7, barheight = 0.5,
                                                  title.position = "left", title.hjust = 1))
            })
            panels <- Filter(Negate(is.null), panels)
            if (length(panels) == 0) next

            n_rows <- max(sapply(panels, function(p)
                nrow(ggplot_build(p)$data[[1]])), na.rm = TRUE)
            fig_h <- max(4.0, 2.0 + 0.55 * min(n_rows, 8))
            options(repr.plot.width = 10 * length(panels), repr.plot.height = fig_h)

            combined <- patchwork::wrap_plots(panels, ncol = length(panels)) +
                patchwork::plot_annotation(
                    title = paste0(ds, " | ", lv, " | ", ct_str),
                    theme = theme(plot.title = element_text(face = "bold",
                                                            size = 12, hjust = 0.5))
                )
            print(combined)
        }
    }
} else {
    cat("No ORA hits\n")
}

## Gene loading plots

In [ ]:
sanitize_stub <- function(x) {
    x <- tolower(x)
    x <- gsub("[^[:alnum:]]+", "_", x)
    x <- gsub("^_|_$", "", x)
    gsub("_+", "_", x)
}

theme_loading_plot <- function(base_size = 11) {
    theme_classic(base_size = base_size) %+replace%
        theme(
            axis.line    = element_line(color = "black", linewidth = 0.4),
            axis.ticks   = element_line(color = "black", linewidth = 0.35),
            axis.text.x  = element_blank(),
            axis.ticks.x = element_blank(),
            axis.title.x = element_blank(),
            plot.title   = element_text(face = "bold", hjust = 0.5, size = base_size + 1),
            plot.margin  = margin(5.5, 28, 5.5, 5.5)
        )
}

make_loading_plot <- function(plot_df, label_df, title_text) {
    ggplot(plot_df, aes(x = rank, y = loading)) +
        geom_point(size = 1.2, color = "grey75") +
        geom_point(data = label_df, color = "#C23B22", size = 1.6) +
        ggrepel::geom_text_repel(
            data             = label_df,
            aes(label        = label),
            size             = 3,
            fontface         = "italic",
            direction        = "y",
            hjust            = 0,
            seed             = 42,
            max.overlaps     = Inf,
            min.segment.length = 0,
            box.padding      = 0.3,
            point.padding    = 0.2,
            nudge_x          = 8,
            segment.color    = "grey60",
            segment.size     = 0.25
        ) +
        coord_cartesian(clip = "off") +
        scale_x_continuous(expand = expansion(mult = c(0.02, 0.18))) +
        labs(title = title_text, y = "Loadings") +
        theme_loading_plot()
}

gene_loadings_list <- lapply(DATASETS, function(ds) {
    ds_rows    <- top_lvs_df[top_lvs_df$dataset == ds, ]
    unique_lvs <- unique(ds_rows$LV)
    if (length(unique_lvs) == 0) return(NULL)

    Z_path <- file.path(MOD_ROOT, ds, "CLAMPfull", "Z.csv")
    Z      <- read.csv(Z_path, row.names = 1, check.names = FALSE)

    lapply(unique_lvs, function(lv) {
        if (!lv %in% colnames(Z)) return(NULL)

        loadings <- Z[[lv]]
        names(loadings) <- rownames(Z)
        n_top    <- max(1L, ceiling(length(loadings) * TOP_GENE_PCT))
        ord      <- order(loadings, decreasing = TRUE)

        df <- data.frame(
            dataset  = ds,
            LV       = lv,
            rank     = seq_len(n_top),
            gene     = rownames(Z)[ord][seq_len(n_top)],
            loading  = loadings[ord][seq_len(n_top)],
            label    = ifelse(seq_len(n_top) <= N_LABEL,
                              rownames(Z)[ord][seq_len(n_top)],
                              NA_character_),
            stringsAsFactors = FALSE
        )

        cts_for_lv <- unique(ds_rows$cell_type[ds_rows$LV == lv])
        ct_str     <- paste(ct_label(cts_for_lv), collapse = ", ")
        max_r      <- max(top_lvs_df$cor[top_lvs_df$dataset == ds & top_lvs_df$LV == lv],
                          na.rm = TRUE)
        title_str  <- sprintf("%s | %s\n%s  (r = %.2f)", ds, lv, ct_str, max_r)

        label_df <- df[!is.na(df$label), ]
        p        <- make_loading_plot(df, label_df, title_str)

        stub <- paste0(sanitize_stub(ds), "_", tolower(lv))
        ggsave(
            filename = file.path(OUT_DIR, "plots", paste0("loading_", stub, ".pdf")),
            plot     = p, width = 5, height = 4, bg = "white"
        )

        options(repr.plot.width = 5, repr.plot.height = 4)
        print(p)

        df
    }) %>% Filter(Negate(is.null), .) %>% do.call(rbind, .)
}) %>% Filter(Negate(is.null), .) %>% do.call(rbind, .)

gene_loadings_df <- gene_loadings_list
cat("Gene loading entries:", nrow(gene_loadings_df), "\n")

## Hard-to-distinguish cell type pairs — biology deep-dive

Five pairs of biologically related subtypes where gene loadings and CellMarker ORA
validate that the CLAMPfull LVs capture the correct biology despite the molecular
similarity between the two cell types.

In [ ]:
# Example pairs — chosen from ORA analysis for strongest cell-type distinction
# pref_db: which database to show in deep-dive bubbles
EXAMPLE_PAIRS <- list(

    # CD14 vs CD16: c7 immunologic signatures distinguish classical vs non-classical monocyte states
    list(ds = "PBMC_1k1k",
         cts = c(CD14_Mono = "LV2",  CD16_Mono = "LV20"),
         pref_db = "c7",
         label = "CD14+ Mono vs CD16+ Mono  [PBMC_1k1k]"),

    # B cell vs Plasma B: c7 captures B cell differentiation / plasmablast states
    list(ds = "PBMC_1k1k",
         cts = c(B_cell = "LV58", Plasma_B = "LV22"),
         pref_db = "c7",
         label = "B cell vs Plasma B  [PBMC_1k1k]"),

    # CD4 vs CD8: c7 immunologic signatures separate T cell lineages
    list(ds = "PBMC_1k1k",
         cts = c(CD4_T = "LV28", CD8_T = "LV4"),
         pref_db = "c7",
         label = "CD4+ T vs CD8+ T  [PBMC_1k1k]"),

    # OPC vs Oli: Allen gives OPALIN oligo (16x) vs OPC PDGFRA sets
    list(ds = "Brain_Xiong2023",
         cts = c(Opc = "LV11", Oli = "LV14"),
         pref_db = "allen",
         label = "OPC vs Oligodendrocyte  [Brain_Xiong2023]"),

    # Ast vs Mic: Allen gives Human Astro FGFR3 (26x) vs Human Micro TYROBP CD74 (79x)
    list(ds = "Brain_Mathys2023",
         cts = c(Ast = "LV17", Mic = "LV6"),
         pref_db = "allen",
         label = "Astrocyte vs Microglia  [Brain_Mathys2023]")
)

cat("Example pairs:\n")
for (ep in EXAMPLE_PAIRS) cat(" •", ep$label, "\n")

Example pairs:
 • CD14+ Mono vs CD16+ Mono  [PBMC_1k1k] 
 • B cell vs Plasma B  [PBMC_1k1k] 
 • CD4+ T vs CD8+ T  [PBMC_1k1k] 
 • OPC vs Oligodendrocyte  [Brain_Xiong2023] 
 • Astrocyte vs Microglia  [Brain_Mathys2023] 


In [ ]:
# ORA bubble plots for example pairs — combined with CLAMP GO-BP pathways
if (!is.null(lv_ora_df) && nrow(lv_ora_df) > 0) {

    db_display <- c(
        "GO-BP" = "GO Biological Process",
        "C7"    = "C7 — Immunological Signatures (MSigDB)",
        "C8"    = "C8 — Cell Type Signatures (MSigDB)",
        "ALLEN" = "Allen Brain Atlas"
    )

    for (ep in EXAMPLE_PAIRS) {
        ds   <- ep$ds
        cts  <- ep$cts
        pref <- ep$pref_db

        # Precompute row counts for height
        n_rows_vec <- sapply(seq_along(cts), function(i) {
            lv      <- cts[i]
            n_clamp <- min(5L, sum(lv_pathways_df$dataset == ds & lv_pathways_df$LV == lv))
            n_ora   <- lv_ora_df %>%
                dplyr::filter(dataset == ds, LV == lv) %>%
                dplyr::group_by(db) %>%
                dplyr::slice_min(padj, n = 8, with_ties = FALSE) %>%
                nrow()
            n_clamp + n_ora
        })

        panels <- lapply(seq_along(cts), function(i) {
            ct <- names(cts)[i]
            lv <- cts[i]

            ct_full <- ct_label(ct)
            cor_val <- top_lvs_df$cor[top_lvs_df$dataset == ds & top_lvs_df$LV == lv]
            cor_val <- if (length(cor_val) > 0) max(cor_val, na.rm = TRUE) else NA_real_

            # CLAMP GO-BP pathways -> source labeled, metric = -log10(FDR)
            clamp_rows <- lv_pathways_df %>%
                dplyr::filter(dataset == ds, LV == lv) %>%
                dplyr::slice_min(FDR, n = 5, with_ties = FALSE) %>%
                dplyr::mutate(
                    source      = unname(db_display["GO-BP"]),
                    term_clean  = clean_pw(pathway),
                    neg_log_p   = pmin(-log10(FDR + 1e-300), 4),
                    bubble_size = 2 + ((AUC - AUC_PATH) / (1 - AUC_PATH)) * 8
                ) %>%
                dplyr::select(source, term_clean, neg_log_p, bubble_size)

            # ORA results — all available databases, metric = -log10(padj)
            ora_rows <- lv_ora_df %>%
                dplyr::filter(dataset == ds, LV == lv) %>%
                dplyr::group_by(db) %>%
                dplyr::slice_min(padj, n = 8, with_ties = FALSE) %>%
                dplyr::ungroup() %>%
                dplyr::mutate(
                    source      = unname(db_display[toupper(db)]),
                    term_clean  = clean_ora_term(term, db),
                    neg_log_p   = pmin(-log10(padj + 1e-300), 4),
                    bubble_size = 2 + pmin(fold_enrichment / 30, 1) * 8
                ) %>%
                dplyr::select(source, term_clean, neg_log_p, bubble_size)

            combined_df <- rbind(clamp_rows, ora_rows)

            if (nrow(combined_df) == 0) {
                return(ggplot() +
                    annotate("text", x = 0.5, y = 0.5,
                             label = paste0(lv, "\n(", ct_full, ")\nno data"),
                             hjust = 0.5, vjust = 0.5, size = 4, color = "grey50") +
                    theme_void() +
                    labs(title = sprintf("%s | %s  (r = %.2f)", lv, ct_full, cor_val)))
            }

            # Facet order: GO-BP first, preferred ORA db second, remaining alphabetically
            gobp_label <- unname(db_display["GO-BP"])
            pref_label <- unname(db_display[toupper(pref)])
            src_order  <- unique(c(gobp_label, pref_label,
                                   sort(setdiff(unique(combined_df$source),
                                                c(gobp_label, pref_label)))))
            src_order          <- src_order[!is.na(src_order) & src_order %in% unique(combined_df$source)]
            combined_df$source <- factor(combined_df$source, levels = src_order)

            # Within each source, sort ascending neg_log_p so most significant lands on top
            combined_df <- combined_df %>%
                dplyr::arrange(source, neg_log_p) %>%
                dplyr::mutate(term_clean = factor(term_clean, levels = unique(term_clean)))

            v_max <- max(2, ceiling(max(combined_df$neg_log_p, na.rm = TRUE)))

            ggplot(combined_df, aes(x = neg_log_p, y = term_clean)) +
                geom_segment(aes(x = 0, xend = neg_log_p, yend = term_clean),
                             color = "#cccccc", linewidth = 0.8, linetype = "dotted") +
                geom_point(aes(color = neg_log_p, size = bubble_size)) +
                scale_color_gradientn(colors = green_colors, limits = c(0, v_max),
                                      name = expression("-" * log[10] ~ "FDR")) +
                scale_size_identity(guide = "none") +
                scale_x_continuous(limits = c(0, v_max + 0.3)) +
                facet_wrap(~ source, scales = "free_y", ncol = 1) +
                labs(x = expression("-" * log[10] ~ "FDR"), y = NULL,
                     title = sprintf("%s | %s  (r = %.2f)", lv, ct_full, cor_val)) +
                bubble_theme +
                theme(
                    strip.background = element_rect(fill = "#f0f0f0", color = NA),
                    strip.text       = element_text(size = 10, face = "bold", hjust = 0.5)
                ) +
                guides(color = guide_colorbar(barwidth = 7, barheight = 0.5,
                                              title.position = "left", title.hjust = 1))
        })

        fig_h <- max(5.0, 2.0 + 0.45 * max(n_rows_vec))
        options(repr.plot.width = 16, repr.plot.height = fig_h)

        combined <- patchwork::wrap_plots(panels, ncol = length(panels)) +
            patchwork::plot_annotation(
                title = ep$label,
                theme = theme(plot.title = element_text(face = "bold", size = 12, hjust = 0.5))
            )
        print(combined)
    }
} else {
    cat("No ORA results available\n")
}

In [ ]:
# Gene loading side-by-side plots for hard-to-distinguish pairs
for (ep in EXAMPLE_PAIRS) {
    ds    <- ep$ds
    cts   <- ep$cts
    label <- ep$label

    Z_path <- file.path(MOD_ROOT, ds, "CLAMPfull", "Z.csv")
    Z      <- read.csv(Z_path, row.names = 1, check.names = FALSE)
    n_genes <- nrow(Z)
    n_top   <- max(1L, ceiling(n_genes * TOP_GENE_PCT))

    panels <- lapply(seq_along(cts), function(i) {
        ct <- names(cts)[i]
        lv <- cts[i]
        if (!lv %in% colnames(Z)) return(NULL)

        loadings <- Z[[lv]]
        names(loadings) <- rownames(Z)
        ord <- order(loadings, decreasing = TRUE)

        df <- data.frame(
            rank    = seq_len(n_top),
            gene    = rownames(Z)[ord][seq_len(n_top)],
            loading = loadings[ord][seq_len(n_top)],
            stringsAsFactors = FALSE
        )
        df$label <- ifelse(df$rank <= N_LABEL, df$gene, NA_character_)
        label_df  <- df[!is.na(df$label), ]

        cor_val <- top_lvs_df$cor[top_lvs_df$dataset == ds & top_lvs_df$LV == lv]
        cor_val <- if (length(cor_val) > 0) max(cor_val, na.rm = TRUE) else NA_real_
        title_str <- sprintf("%s\n%s  (r = %.2f)", lv, ct_label(ct), cor_val)

        make_loading_plot(df, label_df, title_str)
    })
    panels <- Filter(Negate(is.null), panels)
    if (length(panels) == 0) next

    combined <- patchwork::wrap_plots(panels, ncol = length(panels)) +
        patchwork::plot_annotation(
            title = label,
            theme = theme(plot.title = element_text(face = "bold", size = 12, hjust = 0.5))
        )

    options(repr.plot.width = 5 * length(panels), repr.plot.height = 4.5)
    print(combined)
}

## Save outputs

In [ ]:
write.csv(top_lvs_df,       file.path(OUT_DIR, "top_lvs_per_celltype.csv"), row.names = FALSE)
write.csv(lv_pathways_df,   file.path(OUT_DIR, "lv_pathways.csv"),          row.names = FALSE)
write.csv(gene_loadings_df, file.path(OUT_DIR, "gene_loadings.csv"),        row.names = FALSE)

# ORA results (c8 + c7 + Allen Brain) — written to cm_ora path for snakemake compatibility
ora_out <- if (!is.null(lv_ora_df) && nrow(lv_ora_df) > 0) lv_ora_df else
    data.frame(dataset = character(), cell_type = character(), LV = character(),
               db = character(), term = character(), k = integer(), K = integer(),
               M = integer(), N = integer(), fold_enrichment = numeric(),
               pvalue = numeric(), padj = numeric())
write.csv(ora_out, file.path(OUT_DIR, "lv_cellmarker_ora.csv"), row.names = FALSE)

cat("All outputs saved to:", OUT_DIR, "\n")

All outputs saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/02_pseudobulk/01_biology 


## Panel data export (fig5)

In [ ]:
FIG5_PAIRS_KEY <- data.frame(
  dataset = c("PBMC_1k1k",       "PBMC_1k1k",
              "Brain_Xiong2023",  "Brain_Xiong2023",
              "Brain_Mathys2023", "Brain_Mathys2023"),
  LV      = c("LV2",  "LV20", "LV11", "LV14", "LV17", "LV6"),
  stringsAsFactors = FALSE
)

FIG5_DIR <- here("output", "99_panels", "fig5", "fig5")
dir.create(FIG5_DIR, recursive = TRUE, showWarnings = FALSE)

write.csv(
  dplyr::semi_join(top_lvs_df,     FIG5_PAIRS_KEY, by = c("dataset", "LV")),
  file.path(FIG5_DIR, "pair_lv_assignments.csv"), row.names = FALSE
)

write.csv(
  dplyr::semi_join(lv_pathways_df, FIG5_PAIRS_KEY, by = c("dataset", "LV")),
  file.path(FIG5_DIR, "pair_pathways.csv"), row.names = FALSE
)

write.csv(
  dplyr::semi_join(lv_ora_df,      FIG5_PAIRS_KEY, by = c("dataset", "LV")),
  file.path(FIG5_DIR, "pair_ora.csv"), row.names = FALSE
)

cat("fig5 panel data saved to:", FIG5_DIR, "\n")